# Resource-Aware Optimization

Resource-Aware Optimization enables agents to dynamically select the most appropriate (and cost-effective) model or tool for each request. Simple queries route to fast, cheap models; complex reasoning gets the expensive one; real-time queries trigger a search.

## Implementation with Flyte v2

This notebook reimplements the OpenAI prompt-classification + model-routing example from Chapter 16 using **Flyte v2 primitives**.

#### Original vs Flyte v2 — Key Differences

| Aspect | Original (OpenAI script) | Flyte v2 |
|--------|--------------------------|----------|
| **LLM client** | Module-level `OpenAI(api_key=...)` | Task-scoped `AsyncOpenAI` — safe in containers |
| **Routing logic** | Plain function `handle_prompt()` | Flyte task with typed inputs/outputs |
| **Classification caching** | None (re-classifies identical prompts) | `cache="auto"` — identical prompts skip the classifier LLM call |
| **Observability** | `print()` statements | Model used + classification visible as structured task output |
| **Secrets** | `.env` / `dotenv.load_dotenv()` | `flyte.Secret` injected by cluster |
| **Execution** | In-process only | Local or remote (containers on Kubernetes) |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' openai

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret OPENAI_API_KEY --value sk-proj-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from datetime import timedelta

import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="resource-opt-agent", python_version=(3, 12))
    .with_pip_packages("openai>=1.0.0")
)

opt_env = flyte.TaskEnvironment(
    name="resource_opt",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY"),
    ],
)

### 4. Define data models

The original script returned a plain `dict`. Typed dataclasses make routing decisions and results inspectable in the Flyte UI — you can see exactly which model handled each request without parsing logs.

In [ ]:
@dataclass
class Classification:
    """Query classification result from the router LLM."""
    category: str   # "simple" | "reasoning" | "internet_search"
    prompt: str


@dataclass
class RouterResult:
    """Final output: answer with the routing decision recorded."""
    prompt: str
    classification: str
    model_used: str
    answer: str

### 5. Define the classification and generation tasks

**Critical difference from the original:** The original script creates `client = OpenAI(api_key=...)` at **module level**. In a container, `os.getenv("OPENAI_API_KEY")` at import time returns `None` because the secret isn't injected until the task function executes. Always create clients **inside** the task function.

**Caching the classifier:** `classify_task` uses `cache="auto"`. For identical prompts, Flyte returns the cached classification without calling the LLM — reducing both latency and cost when the same prompt appears across multiple runs.

In [ ]:
CLASSIFIER_SYSTEM = (
    "You are a classifier. Analyze user prompts and respond with exactly one JSON object:\n\n"
    '{ "classification": "<category>" }\n\n'
    "Categories:\n"
    "- simple: direct factual questions requiring no reasoning or current events\n"
    "- reasoning: logic, math, or multi-step inference questions\n"
    "- internet_search: questions about current events or recent data not in training data\n\n"
    "Respond ONLY with the JSON object, no other text."
)

# Model routing table — change these to adjust cost/quality tradeoffs
MODEL_ROUTING = {
    "simple": "gpt-4o-mini",      # fast, cheap
    "reasoning": "o4-mini",        # reasoning-optimized
    "internet_search": "gpt-4o",   # capable, with injected search context
}


@opt_env.task(cache="auto")
async def classify_task(prompt: str) -> Classification:
    """
    Classify prompt complexity to select the appropriate model.

    cache="auto": identical prompts are classified once and cached —
    no redundant LLM calls for repeated inputs.
    """
    import json
    from openai import AsyncOpenAI

    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": CLASSIFIER_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    content = (response.choices[0].message.content or "{}").strip()
    data = json.loads(content)
    category = data.get("classification", "simple")
    if category not in MODEL_ROUTING:
        category = "simple"
    return Classification(category=category, prompt=prompt)


@opt_env.task(cache=flyte.Cache(behavior="disable"))
async def generate_task(classification: Classification) -> RouterResult:
    """
    Generate response using the model appropriate for the query complexity.

    Routes:
      simple        → gpt-4o-mini   (fast, cheap)
      reasoning     → o4-mini       (reasoning-optimized)
      internet_search → gpt-4o      (with simulated search context)
    """
    from openai import AsyncOpenAI

    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
    model = MODEL_ROUTING.get(classification.category, "gpt-4o-mini")

    if classification.category == "internet_search":
        # In production, inject real search results here
        user_content = (
            f"[Note: No live search results available in this example. "
            f"Answer based on training data if possible.]\n\n"
            f"Query: {classification.prompt}"
        )
    else:
        user_content = classification.prompt

    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_content}],
        temperature=1,
    )
    answer = (response.choices[0].message.content or "").strip()

    return RouterResult(
        prompt=classification.prompt,
        classification=classification.category,
        model_used=model,
        answer=answer,
    )

### 6. Orchestrate the router pipeline

In [ ]:
@opt_env.task(cache=flyte.Cache(behavior="disable"))
async def handle_prompt(prompt: str) -> RouterResult:
    """Classify and route a user prompt to the appropriate model."""
    classification = await classify_task(prompt=prompt)
    return await generate_task(classification=classification)

### 7. Run locally

In [ ]:
TEST_PROMPTS = [
    "What is the capital of Australia?",                           # simple
    "Explain the impact of quantum computing on cryptography.",     # reasoning
    "What were the top AI announcements at Google I/O 2025?",      # internet_search
]

for prompt in TEST_PROMPTS:
    run = flyte.run(handle_prompt, prompt=prompt)
    run.wait()
    result: RouterResult = run.outputs()[0]
    print(f"Prompt:    {result.prompt[:60]}...")
    print(f"Category:  {result.classification}")
    print(f"Model:     {result.model_used}")
    print(f"Answer:    {result.answer[:120]}...")
    print()

### Running remotely

With `cache="auto"` on `classify_task`, running the same prompt twice hits the cache on the second call — zero classification cost. The `RouterResult` dataclass appears as structured output in the Flyte UI, making the routing decision fully auditable.

In [ ]:
run = flyte.run(handle_prompt, prompt="What is the capital of Australia?")
run.wait()
result = run.outputs()[0]
print(f"Classification: {result.classification}, Model: {result.model_used}")
print(result.answer)

## Scaling the pattern

For high-throughput routing (e.g., production API gateway), use `ReusePolicy` to keep warm classifier pods. The classification call is the hot path — cold-starting a container for every request would dominate latency.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
production_opt_env = flyte.TaskEnvironment(
    name="resource_opt_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="2Gi"),
    secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 16),
        concurrency=8,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)